# 🏙️ City Energy Consumption Analysis & Prediction System

A complete pipeline for generating, analysing, visualising, and predicting
daily electricity usage across 5 city zones.

**Pipeline steps**
1. Synthetic data generation (365 days × 5 zones)
2. Preprocessing & feature engineering
3. Exploratory data analysis
4. Visualisations (monthly trends, correlation heatmap, event comparison)
5. ML model training (Linear Regression + Random Forest)
6. Interactive prediction interface

In [ ]:
# Install requirements (uncomment if needed)
# !pip install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
import warnings
warnings.filterwarnings('ignore')

# Import our module
import energy_analysis as ea

print('✅ Libraries loaded')
print(f'   pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}')

---
## 1. Data Generation

In [ ]:
raw_df = ea.generate_dataset(seed=42)
print(f'Shape : {raw_df.shape}')
print(f'Dates : {raw_df["Date"].min().date()}  →  {raw_df["Date"].max().date()}')
raw_df.head(10)

In [ ]:
print('Missing values:\n', raw_df.isnull().sum())
print('\nData types:\n', raw_df.dtypes)
raw_df.describe().round(1)

---
## 2. Preprocessing & Feature Engineering

In [ ]:
df = ea.preprocess(raw_df)
print('New feature columns:', [c for c in df.columns if c not in raw_df.columns])
df[['Date','ZoneID','Month','DayOfWeek','IsWeekend','ZoneEncoded']].head()

In [ ]:
# Per-zone summary statistics
df.groupby('ZoneID')['EnergyConsumption'].describe().round(0)

---
## 3. Exploratory Analysis

In [ ]:
# Monthly averages
monthly = ea.monthly_zone_summary(df)
monthly_pivot = monthly.pivot(index='Month', columns='ZoneID', values='AvgConsumption').round(0)
monthly_pivot

In [ ]:
# Correlation analysis
corr = ea.correlation_matrix(df)
print('Correlations with EnergyConsumption:')
corr['EnergyConsumption'].drop('EnergyConsumption').sort_values(key=abs, ascending=False)

In [ ]:
# Event impact
ev = ea.event_vs_nonevent(df)
ev_pivot = ev.pivot(index='ZoneID', columns='SpecialEvent', values='EnergyConsumption')
ev_pivot.columns = ['No Event (kWh)', 'Special Event (kWh)']
ev_pivot['Uplift (kWh)'] = ev_pivot['Special Event (kWh)'] - ev_pivot['No Event (kWh)']
ev_pivot['Uplift (%)']   = (ev_pivot['Uplift (kWh)'] / ev_pivot['No Event (kWh)'] * 100).round(1)
ev_pivot.round(0)

---
## 4. Visualisations

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

# Plot 1 — Monthly energy trends
ea.plot_monthly_trends(df, 'outputs/plot1_monthly_trends.png')
Image('outputs/plot1_monthly_trends.png')

In [ ]:
# Plot 2 — Correlation heatmap
ea.plot_correlation_heatmap(df, 'outputs/plot2_correlation_heatmap.png')
Image('outputs/plot2_correlation_heatmap.png')

In [ ]:
# Plot 3 — Event vs Non-event
ea.plot_event_vs_nonevent(df, 'outputs/plot3_event_comparison.png')
Image('outputs/plot3_event_comparison.png')

---
## 5. Machine Learning Models

In [ ]:
rf, lr, X_test, y_test, z_test, results = ea.train_models(df)

print('Model Evaluation (held-out 20% test set)')
print('─' * 40)
print(f'Linear Regression  MAE : {results["lr_mae"]:>10,.0f} kWh')
print(f'Random Forest      MAE : {results["rf_mae"]:>10,.0f} kWh  ← best')

In [ ]:
# Feature importance
import pandas as pd
fi = pd.Series(rf.feature_importances_, index=ea.FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#1A1D27')
bars = ax.barh(fi.index, fi.values, color='#4F8EF7', edgecolor='#0F1117')
ax.set_xlabel('Importance', color='#E8EAF0')
ax.set_title('Random Forest — Feature Importances', color='#E8EAF0', pad=12)
ax.tick_params(colors='#E8EAF0')
for spine in ax.spines.values(): spine.set_edgecolor('#262A38')
ax.grid(axis='x', color='#262A38', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter
ea.plot_model_performance(y_test, results['rf_pred'], z_test,
                          'outputs/plot4_model_performance.png')
Image('outputs/plot4_model_performance.png')

---
## 6. Interactive Prediction

Run `python energy_analysis.py` from the terminal for the full CLI.

Below is a programmatic example:

In [ ]:
import datetime

def predict(zone_name, temperature, humidity, special_event):
    """Wrapper for single prediction."""
    le_zones = {zone: i for i, zone in enumerate(sorted(ea.ZONES))}
    tomorrow  = datetime.date.today() + datetime.timedelta(days=1)
    X = pd.DataFrame([{
        'ZoneEncoded':    le_zones[zone_name],
        'AvgTemperature': temperature,
        'Humidity':       humidity,
        'SpecialEvent':   special_event,
        'Month':          tomorrow.month,
        'DayOfWeek':      tomorrow.weekday(),
        'IsWeekend':      int(tomorrow.weekday() >= 5),
    }])
    pred = rf.predict(X)[0]
    print(f'Zone: {zone_name:<20}  Temp: {temperature}°C  Humidity: {humidity}%  '
          f'Event: {bool(special_event)}')
    print(f'  ⚡ Predicted Consumption: {pred:,.0f} kWh  ({pred/1000:.2f} MWh)\n')
    return pred

# Example predictions
predict('Z4_Commercial',  28.5, 60, 1)   # hot summer day with festival
predict('Z2_Industrial',  10.0, 70, 0)   # cold winter day, no event
predict('Z3_Residential', 20.0, 55, 0)   # mild day, no event

---
## Key Insights

| Insight | Detail |
|---|---|
| **Zone size** | Industrial (Z2) and Airport (Z5) zones consume 2–3× more than residential |
| **Weekend effect** | Consumption drops ~15% on weekends (weekday factor encoded) |
| **Special events** | Add 8–23% uplift depending on zone; Commercial highest impact |
| **Temperature** | U-shaped relationship — extremes (cold/hot) raise demand |
| **Best model** | Random Forest (MAE ≈ 695 kWh) vs Linear Regression (MAE ≈ 9,940 kWh) |
| **Top feature** | `ZoneEncoded` — zone identity dominates baseline consumption |
